**Enhancing Search Engine Relevance for Video Subtitles**

In [5]:
import sqlite3
import zipfile
import io

# Connect to the subtitles database
db_path = r'/content/drive/MyDrive/Copy of eng_subtitles_database.db'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Fetch a single subtitle record
cursor.execute("SELECT num, name, content FROM zipfiles LIMIT 1")
num, name, content = cursor.fetchone()

# Extract .srt from zipped binary
zip_file = zipfile.ZipFile(io.BytesIO(content))
srt_file = zip_file.namelist()[0]  # Assume 1 file per zip
text = zip_file.read(srt_file).decode('utf-8', errors='ignore')

print(f"Subtitle ID: {num}\nFilename: {name}\nText Preview:\n{text[:1000]}")


Subtitle ID: 9180533
Filename: the.message.(1976).eng.1cd
Text Preview:
1
00:00:06,000 --> 00:00:12,074
Watch any video online with Open-SUBTITLES
Free Browser extension: osdb.link/ext

2
00:02:26,198 --> 00:02:29,953
In the name of God, the most gracious, the most Merciful.

3
00:02:31,072 --> 00:02:33,370
From Muhammad, the Messenger of God

4
00:02:33,550 --> 00:02:36,047
to Heraclius, the emperor of Byzantium.

5
00:02:36,407 --> 00:02:39,464
greetings to him who is the
follower of righteous guidance.

6
00:02:39,783 --> 00:02:42,591
I bid you to hear the divine call.

7
00:02:43,160 --> 00:02:45,817
I am the messenger of God to the people;

8
00:02:46,337 --> 00:02:48,784
accept Islam for your salvation.

9
00:02:52,231 --> 00:02:54,709
He speaks of a new prophet in Arabia.

10
00:02:55,068 --> 00:02:57,825
Was it like this when John, the Baptist
came to king Herod

11
00:02:58,145 --> 00:03:01,272
out of the desert, crying about salvation?

12
00:03:26,136 --> 00:03:28,903
To Muq

Data preprocessing

In [6]:
import re

def clean_subtitle_text(raw_text):
    text = re.sub(r'\d{2}:\d{2}:\d{2},\d{3} --> \d{2}:\d{2}:\d{2},\d{3}', '', raw_text)
    text = re.sub(r'^\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'http\S+|www\S+|osdb\.link\S+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

cleaned_text = clean_subtitle_text(text)
print(cleaned_text[:1000])


watch any video online with open-subtitles free browser extension: in the name of god, the most gracious, the most merciful. from muhammad, the messenger of god to heraclius, the emperor of byzantium. greetings to him who is the follower of righteous guidance. i bid you to hear the divine call. i am the messenger of god to the people; accept islam for your salvation. he speaks of a new prophet in arabia. was it like this when john, the baptist came to king herod out of the desert, crying about salvation? to muqawqis, patriarch of alexandria. kisra, emperor of persia. muhammad calls you with the call of god. accept islam for your salvation... embrace islam. you come out of the desert, smelling of camel and goat. to tell persia where he should kneel? muhammad, messenger of god. who gave him this authority? god sent muhammad as a mercy to mankind. the scholars and historians of islam - the university of al-azhar in cairo the high islamic congress of the shiat in lebanon the makers of this

Chunking

In [7]:
def chunk_text(text, chunk_size=50, overlap=10):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if len(chunk.split()) >= 10:
            chunks.append(chunk)
    return chunks

chunks = chunk_text(cleaned_text)
print(f"Chunks: {len(chunks)}\n\nSample:\n{chunks[0]}")

Chunks: 227

Sample:
watch any video online with open-subtitles free browser extension: in the name of god, the most gracious, the most merciful. from muhammad, the messenger of god to heraclius, the emperor of byzantium. greetings to him who is the follower of righteous guidance. i bid you to hear the divine call.


Embedding

In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = model.encode(chunks, show_progress_bar=True)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Installation chromadb

In [9]:
import chromadb

# Use the new default in-memory client (sufficient for local/dev)
client = chromadb.Client()

# Create collection
collection = client.get_or_create_collection("subtitle_chunks")

# Add chunks and embeddings
ids = [f"chunk_{i}" for i in range(len(chunks))]

collection.add(
    documents=chunks,
    ids=ids,
    embeddings=chunk_embeddings
)

print("✅ Chunks added to in-memory ChromaDB")



✅ Chunks added to in-memory ChromaDB


Audio Transcription

In [10]:
import whisper

# Load Whisper model
whisper_model = whisper.load_model("base")

# Transcribe audio file
result = whisper_model.transcribe(r"/content/Sample audio.m4a")
transcribed_text = result['text']
print("🎧 Transcribed:\n", transcribed_text)

/usr/local/lib/python3.11/dist-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


🎧 Transcribed:
  In the name of God, the most gracious, the most merciful, from Muhammad, the messenger of God to Heraklius, the Emperor Bazzanium, greetings to him who is a follower of righteous guidance. I bid you to hear the divine call.


In [11]:
query_cleaned = clean_subtitle_text(transcribed_text)
print("🧹 Cleaned Query:\n", query_cleaned)
query_embedding = model.encode([query_cleaned])[0]


🧹 Cleaned Query:
 in the name of god, the most gracious, the most merciful, from muhammad, the messenger of god to heraklius, the emperor bazzanium, greetings to him who is a follower of righteous guidance. i bid you to hear the divine call.


In [12]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5  # Top 5 closest chunks
)

# Display results
print("🔍 Top Matching Subtitle Chunks:\n")
for doc, dist in zip(results['documents'][0], results['distances'][0]):
    print(f"[Score: {1 - dist:.4f}] {doc}\n")

🔍 Top Matching Subtitle Chunks:

[Score: 0.6924] watch any video online with open-subtitles free browser extension: in the name of god, the most gracious, the most merciful. from muhammad, the messenger of god to heraclius, the emperor of byzantium. greetings to him who is the follower of righteous guidance. i bid you to hear the divine call.

[Score: 0.2201] righteous guidance. i bid you to hear the divine call. i am the messenger of god to the people; accept islam for your salvation. he speaks of a new prophet in arabia. was it like this when john, the baptist came to king herod out of the desert, crying about

[Score: 0.1737] letters, from muhammad, messenger of god, to the rulers of the world... ...call the world to islam! to heraclius, emperor of byzantium, kisra, emperor of persia, muqawqis, patriarch of alexandria. god go with you! god is great! god is great! god is great! there are no different races in islam.

[Score: 0.1342] eager to listen to him. we will begin with the weak

In [26]:
# Create a Streamlit script
script_path = "/content/whisper_transcribe.py"
script_content = """
import streamlit as st
import whisper
import chromadb
from sentence_transformers import SentenceTransformer

st.title("🎙️ Audio Transcription")

# Load Whisper Model
@st.cache_resource
def load_whisper():
    return whisper.load_model("base")

whisper_model = load_whisper()

# Load Sentence Transformer
@st.cache_resource
def load_embedding_model():
    return SentenceTransformer("all-MiniLM-L6-v2")

embedding_model = load_embedding_model()

# Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path="chroma_db")
collection = chroma_client.get_or_create_collection(name="subtitles")

# Sample subtitle chunks (Replace with real data)
subtitles = [
    "The Prophet Muhammad sent letters to rulers inviting them to Islam.",
    "Heraklius was a Byzantine Emperor who received the Prophet’s letter.",
    "The letter to Heraklius spoke about the oneness of God and guidance.",
    "Islamic history contains many diplomatic correspondences.",
    "Studying old letters helps us understand history better."
]

# Store subtitles in ChromaDB with embeddings
for i, subtitle in enumerate(subtitles):
    embedding = embedding_model.encode(subtitle).tolist()
    collection.add(ids=[str(i)], embeddings=[embedding], documents=[subtitle])

# File uploader
uploaded_file = st.file_uploader("Upload an audio file", type=["wav", "mp3", "m4a"])

if uploaded_file is not None:
    with open("temp_audio.m4a", "wb") as f:
        f.write(uploaded_file.getbuffer())

    st.audio(uploaded_file, format='audio/mpeg')

    with st.spinner("Transcribing..."):
        result = whisper_model.transcribe("temp_audio.m4a")
        transcribed_text = result['text']
        st.write("### 🎧 Transcribed Text:")
        st.write(transcribed_text)

        # Generate embedding for transcribed text
        query_embedding = embedding_model.encode([transcribed_text])[0].tolist()

        # Query top 5 matching subtitles
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=5  # Top 5 closest chunks
        )

        st.write("### 🔍 Top Matching Subtitle Chunks:")
        for doc, dist in zip(results['documents'][0], results['distances'][0]):
            st.write(f"**[Score: {1 - dist:.4f}]** {doc}")

"""

# Save the script
with open(script_path, "w") as script_file:
    script_file.write(script_content)

print("✅ app.py created successfully!")


✅ app.py created successfully!


In [27]:
# Run the Streamlit app
!streamlit run /content/whisper_transcribe.py &>/content/logs.txt &

# Expose via localtunnel
!npx localtunnel --port 8501 & curl ipv4.icanhazip.com

35.231.88.234
⠙⠹⠸⠼⠴⠦your url is: https://rude-spiders-grow.loca.lt
